In [ ]:
import pandas as pd
import plotly.express as px
from message_ix_models.util import package_data_path
from itertools import combinations
from scgraph.geographs.marnet import marnet_geograph
import json
import requests
import folium 

from message_ix_models.project.weu_security.reporting.maps.base_regions import base_regions_map

# Read in port data
ports = pd.read_excel(package_data_path('bilateralize', 'distances', 'distances.xlsx'), sheet_name='node_ports')
ports = ports[ports['Regionalization'] == 'R12']

# Remove rows with missing coordinates
ports_clean = ports.dropna(subset=["Latitude", "Longitude"])

if ports_clean.empty:
    raise ValueError("No valid coordinate data found in the file")

# Get all combinations of ports (without repetition)
port_combinations = list(combinations(ports_clean.index, 2))

# Calculate distances between all port combinations
distances = []
for i, j in port_combinations:
        port1 = ports_clean.iloc[i]
        port2 = ports_clean.iloc[j]

        distance = marnet_geograph.get_shortest_path(
            origin_node = {"latitude": port1["Latitude"], 
                           "longitude": port1["Longitude"]},
            destination_node = {"latitude": port2["Latitude"], 
                                "longitude": port2["Longitude"]},
        )

        distances.append(
            {
                "Port1": port1["Port"],
                "Port2": port2["Port"],
                "Node1": port1["Node"],
                "Node2": port2["Node"],
                "Distance_km": distance['length'],
                "Path": distance['coordinate_path']
            }
        )

# Create DataFrame with results
ports_path = pd.DataFrame(distances)

In [ ]:
# Set up base map
map = base_regions_map()

# Folium does not wrap coordinates around the globe, so we need to adjust the path accordingly
# Essentially we create 5 differnt paths that offset the original path by 0, 360, -360, 720, -720 degrees
# This allows the illusion that the path wraps around the globe with folium
def adjustArcPath(path):
    for index in range(1, len(path)):
        x = path[index][1]
        prevX = path[index - 1][1]
        path[index][1] = x - (round((x - prevX)/360,0) * 360)
    return path

def modifyArcPathLong(points, amount):
    return [[i[0], i[1]+amount] for i in points]

def getCleanArcPath(path):
    path = adjustArcPath(path)
    return [
        path,
        modifyArcPathLong(path, 360),
        modifyArcPathLong(path, -360),
        modifyArcPathLong(path, 720),
        modifyArcPathLong(path, -720)
    ]
    
# Populate it with port path
for c in range(0, len(ports_path)):
    folium.PolyLine(
        getCleanArcPath(ports_path['Path'].iloc[c]),
        color='darkblue',
        weight=2,
        opacity=0.5,
        popup='Length (KM): ' + str(ports_path['Distance_km'].iloc[c])
    ).add_to(map)

# Populate with port markers
for c in range(0, len(ports)):
    folium.CircleMarker(location=[ports['Latitude'].iloc[c],
                                  ports['Longitude'].iloc[c]],
                        color = 'darkblue',
                        radius=4,
                        weight=5
                       ).add_to(map)


In [ ]:
map